# 03. 다중 로봇 작업 배정과 안전 shield

## 목표
로봇별 능력과 비용을 이용해 작업을 배정하고, 학습 정책이 제안한 행동을 독립적인 안전 규칙으로 승인·수정·거부합니다. 실제 Gemini Robotics API를 호출하지 않는 toy orchestration입니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Robot:
    name: str
    skills: frozenset
    travel_cost: int

robots = [
    Robot('mobile_humanoid', frozenset({'walk', 'floor_pick', 'carry'}), 2),
    Robot('fixed_dual_arm', frozenset({'precise_insert', 'pack'}), 0),
    Robot('dexterous_hand', frozenset({'tie', 'zip'}), 1),
]
tasks = [('collect_floor', 'floor_pick'), ('insert_part', 'precise_insert'), ('seal_bag', 'zip')]

assignments = {}
for task, required_skill in tasks:
    candidates = [r for r in robots if required_skill in r.skills]
    assignments[task] = min(candidates, key=lambda r: r.travel_cost).name if candidates else 'human_help'
print(assignments)

In [ ]:
def safety_shield(action):
    # 정책과 독립적으로 검사해야 모델 오류 하나가 바로 물리 사고가 되지 않습니다.
    if action['human_distance_m'] < 0.8:
        return {'decision': 'STOP', 'reason': '사람이 보호 거리 안에 있음'}
    if action['force_n'] > 35:
        clipped = dict(action)
        clipped['force_n'] = 35
        return {'decision': 'MODIFY', 'action': clipped, 'reason': '힘 상한 적용'}
    if action['confidence'] < 0.6:
        return {'decision': 'ASK_HUMAN', 'reason': '상황 불확실'}
    return {'decision': 'ALLOW', 'action': action}

proposals = [
    {'name': 'reach', 'human_distance_m': 1.5, 'force_n': 12, 'confidence': 0.9},
    {'name': 'push', 'human_distance_m': 1.2, 'force_n': 50, 'confidence': 0.8},
    {'name': 'walk', 'human_distance_m': 0.4, 'force_n': 0, 'confidence': 0.95},
    {'name': 'grasp_unknown', 'human_distance_m': 2.0, 'force_n': 10, 'confidence': 0.4},
]
for proposal in proposals:
    print(proposal['name'], '=>', safety_shield(proposal))

## 설계 토론

1. 중앙 allocator가 멈추면 각 로봇의 lease가 만료되어 안전 정지하도록 heartbeat를 추가하세요.
2. 두 로봇이 같은 통로를 예약하지 못하도록 shared-resource lock을 설계하세요.
3. 규칙 기반 shield도 센서 오류에 취약합니다. 이중 센서, 안전 PLC와 물리 비상 정지를 별도 계층으로 두어야 합니다.
4. success rate뿐 아니라 근접 정지 횟수, 사람 개입률, task latency와 recovery 횟수를 함께 기록하세요.